# Python Functions

---

## 1. Function Syntax

A function is a reusable block of code that performs a specific task.

```python
def greet(name: str) -> str:
    """Returns a greeting message."""
    return f"Hello, {name}!"

greet("Alice")  # "Hello, Alice!"
```

---

## 2. Lambda Syntax

A lambda is an anonymous, single-expression function.

```python
square = lambda x: x ** 2
square(5)  # 25
```

**Syntax:** `lambda parameters: expression`

---

## 3. Parameter Types

| Type | Example | Description |
|------|---------|-------------|
| Positional | `def f(a, b)` | Order matters |
| Named/Keyword | `f(a=1, b=2)` | Order-independent |
| Default | `def f(a=10)` | Fallback value |
| `*args` | `def f(*args)` | Variable positional args (tuple) |
| `**kwargs` | `def f(**kwargs)` | Variable keyword args (dict) |
| Keyword-only | `def f(*, k)` | Must be passed by name |

```python
def demo(pos, /, normal, *, kw_only, **kwargs):
    print(pos, normal, kw_only, kwargs)

demo(1, 2, kw_only=3, extra=4)
# 1 2 3 {'extra': 4}
```

---

## 4. Lambda Use Cases

Lambdas shine as short, throwaway functions — especially as arguments.

```python
nums = [3, 1, 4, 1, 5]
sorted(nums, key=lambda x: -x)   # [5, 4, 3, 1, 1]

pairs = [(1, 'b'), (2, 'a')]
sorted(pairs, key=lambda p: p[1]) # [(2, 'a'), (1, 'b')]
```

> ⚠️ Avoid lambdas for complex logic — use `def` for readability.

---

## 5. `map`, `filter`, `zip`, `reduce`

### `map(func, iterable)` — Transform each element
```python
list(map(lambda x: x * 2, [1, 2, 3]))  # [2, 4, 6]
```

### `filter(func, iterable)` — Keep elements where func is True
```python
list(filter(lambda x: x % 2 == 0, [1, 2, 3, 4]))  # [2, 4]
```

### `zip(*iterables)` — Pair elements from multiple iterables
```python
list(zip([1, 2], ['a', 'b']))  # [(1, 'a'), (2, 'b')]
```

### `reduce(func, iterable)` — Accumulate to a single value
```python
from functools import reduce
reduce(lambda a, b: a + b, [1, 2, 3, 4])  # 10
```

---

## 6. Scope — Definition

**Scope** defines where a variable is accessible in code. Python resolves names using the **LEGB rule**:

| Letter | Scope | Description |
|--------|-------|-------------|
| **L** | Local | Inside the current function |
| **E** | Enclosing | In any enclosing (outer) function |
| **G** | Global | Module-level |
| **B** | Built-in | Python's built-in names (`len`, `print`…) |

```python
x = "global"

def outer():
    x = "enclosing"
    def inner():
        x = "local"
        print(x)  # "local"
    inner()
```

---

## 7. Local, Nonlocal, Global, and Enclosing Scopes

```python
count = 0          # global

def outer():
    total = 10     # enclosing for inner()

    def inner():
        local_val = 5  # local
        nonlocal total
        total += local_val  # modifies enclosing
        global count
        count += 1         # modifies global
    inner()
    print(total)   # 15

outer()
print(count)       # 1
```

---

## 8. `global` and `nonlocal` Keywords

- **`global`** — Declares that a variable refers to the module-level name.
- **`nonlocal`** — Declares that a variable refers to the nearest enclosing (non-global) scope.

### Real-world use cases

**`global` — shared counter/config:**
```python
_cache = {}

def add_to_cache(key, value):
    global _cache
    _cache[key] = value
```

**`nonlocal` — stateful closure (e.g., factory):**
```python
def make_counter():
    count = 0
    def increment():
        nonlocal count
        count += 1
        return count
    return increment

counter = make_counter()
counter()  # 1
counter()  # 2
```

---

## 9. Scopes in Nested Functions & Closures

A **closure** is an inner function that remembers variables from its enclosing scope even after the outer function has returned.

```python
def multiplier(factor):
    def multiply(x):
        return x * factor   # 'factor' is captured (closed over)
    return multiply

double = multiplier(2)
double(5)   # 10
double(9)   # 18
```

The variable `factor` lives in the closure's `__closure__` attribute, not in any active stack frame.

---

## 10. `globals()` and `locals()`

- **`globals()`** — Returns the module-level symbol table as a dict.
- **`locals()`** — Returns the current local symbol table as a dict.

```python
x = 42
print(globals()['x'])   # 42

def demo():
    a = 1
    print(locals())     # {'a': 1}
demo()
```

### Can we mutate them?

- **`globals()`** — ✅ Yes, mutations affect the actual global namespace.
- **`locals()`** — ❌ No (generally). Modifying the returned dict does **not** change local variables; the dict is a snapshot copy inside functions.

```python
globals()['new_var'] = 99
print(new_var)   # 99  ✅

def test():
    a = 1
    locals()['a'] = 999
    print(a)     # still 1 ❌
```

---

## 11. Exception Variable Scope

After a `try/except` block, the exception variable is **deleted** from scope — even if it was defined before.

```python
e = "original"
try:
    raise ValueError("oops")
except ValueError as e:
    print(e)   # "oops"

print(e)       # NameError: name 'e' is not defined
```

> This is by design in Python 3: the exception variable is cleaned up after the `except` block to avoid reference cycles. Save it explicitly if needed:

```python
try:
    raise ValueError("oops")
except ValueError as e:
    saved = e   # save before it's deleted

print(saved)   # "oops" ✅
```

---

## 12. Functions as First-Class Citizens

In Python, functions are objects. They can be:

```python
# As a variable
say_hi = lambda: "Hi!"

# As a parameter
def apply(func, value):
    return func(value)
apply(str.upper, "hello")   # "HELLO"

# As a return value
def get_greeter(lang):
    if lang == "es": return lambda n: f"Hola, {n}!"
    return lambda n: f"Hello, {n}!"

greet_es = get_greeter("es")
greet_es("Ana")   # "Hola, Ana!"
```

---

## 13. Higher-Order Functions

A **higher-order function** either takes a function as an argument or returns one.

```python
# Decorator as a higher-order function
def logger(func):
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

@logger
def add(a, b):
    return a + b

add(2, 3)
# Calling add
# 5
```

Other examples: `map`, `filter`, `sorted(key=...)`, `functools.partial`.

---

## 14. Function Meta Information

### Docstrings
```python
def add(a: int, b: int) -> int:
    """
    Adds two integers.

    Args:
        a: First number.
        b: Second number.

    Returns:
        Sum of a and b.
    """
    return a + b

help(add)        # prints the docstring
add.__doc__      # access it programmatically
```

### Type Hints
```python
from typing import Optional, list

def find(items: list[str], target: str) -> Optional[int]:
    ...
```

Type hints don't enforce types at runtime but improve readability and enable static analysis (e.g., `mypy`).

### Other Metadata
```python
add.__name__       # "add"
add.__module__     # module name
add.__annotations__ # {'a': int, 'b': int, 'return': int}
```

---

## 15. Recursion

A **recursive function** calls itself to solve a smaller version of the same problem.

Every recursive function needs:
1. **Base case (termination point)** — stops the recursion.
2. **Recursive step** — reduces the problem toward the base case.

```python
def factorial(n: int) -> int:
    if n == 0:          # base case
        return 1
    return n * factorial(n - 1)  # recursive step

factorial(5)  # 120
```

---

## 16. Recursion — Pros & Cons

| ✅ Pros | ❌ Cons |
|--------|--------|
| Elegant for tree/graph traversal | Risk of `RecursionError` (default limit ~1000) |
| Mirrors mathematical definitions | Higher memory usage (call stack) |
| Simplifies divide-and-conquer problems | Harder to debug |
| Clean code for self-similar problems | Slower than iteration in Python (no TCO) |

---

## 17. Recursion Use Cases

| Use Case | Example |
|----------|---------|
| Tree traversal | Walking a file system, DOM |
| Sorting | Merge sort, quick sort |
| Mathematical | Fibonacci, factorial, GCD |
| Backtracking | Maze solving, N-Queens |
| Parsing | Recursive descent parsers |

```python
# Flatten nested list recursively
def flatten(lst):
    result = []
    for item in lst:
        if isinstance(item, list):
            result.extend(flatten(item))  # recursive step
        else:
            result.append(item)           # base case
    return result

flatten([1, [2, [3, 4]], 5])  # [1, 2, 3, 4, 5]
```
